## SECTION 1 — Setup

In [ ]:
import random
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image

from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
import torchvision.transforms as T

import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# Set seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PROJECT_ROOT = Path("..").resolve()

## SECTION 2 — Load Train / Val Splits from CSV

In [ ]:
# Load splits generated by 01_eda.ipynb (Phase 2)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")

# Resolve relative paths to absolute for image loading
train_df["image_path_abs"] = train_df["image_path"].apply(lambda p: str(PROJECT_ROOT / p))
val_df["image_path_abs"] = val_df["image_path"].apply(lambda p: str(PROJECT_ROOT / p))

# Class info
num_classes = train_df["encoded_label"].nunique()
class_names = sorted(train_df["label"].unique())

print(f"Train: {len(train_df):,} images")
print(f"Val:   {len(val_df):,} images")
print(f"Classes: {num_classes}")
train_df.head()

## SECTION 3 — Image Transforms

**Data Augmentation Strategy:**
We apply `RandomResizedCrop(224)`, `RandomHorizontalFlip()`, `RandomRotation(15)`, and `ColorJitter` during training to combat overfitting, simulating variations in how photos are taken. Normalization uses ImageNet means and standard deviations. Validation just resizes to 256 and CenterCrops to 224 to ensure consistent evaluation.

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = T.Compose([
    T.RandomResizedCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_tfms = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

## SECTION 4 — Custom Dataset

In [ ]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Handle both relative and absolute paths
        path_obj = Path(img_path)
        if not path_obj.is_absolute():
            path_obj = PROJECT_ROOT / img_path
        
        img = Image.open(path_obj).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

## SECTION 5 — Baseline CNN Architecture

**Tensor Flow:**
Input Shape: `(Batch, Channels, Height, Width)` -> `(B, 3, 224, 224)`

After Conv 1: `(B, 32, 112, 112)`

After Conv 2: `(B, 64, 56, 56)`

After Conv 3: `(B, 128, 28, 28)`

After AdaptiveAvgPool: `(B, 128, 1, 1)`

Flatten: `(B, 128)`

Linear Classifier: `(B, num_classes)`

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
    
    def forward(self, x):
        return self.block(x)

class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model_test = BaselineCNN(num_classes=num_classes)
print(model_test)

## SECTION 6 — Class Imbalance Handling

The dataset is extremely imbalanced. To prevent the model from prioritizing majority classes, we apply class weighting within `CrossEntropyLoss`.

*Important: We compute these class weights INSIDE the cross-validation loop using only the `train_df` labels for each fold to strictly isolate the validation data.*

## SECTION 8 \& SECTION 9 — DataLoaders & Training Loop Def

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        with autocast(device_type='cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation", leave=False):
            images, labels = images.to(device), labels.to(device)
            
            with autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

## SECTION 10 — Training Execution (Single Split, AMP)

**Config:** EPOCHS=10, BATCH_SIZE=32, LR=1e-3, mixed precision enabled.
Best model saved by val macro-F1.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 32
NUM_WORKERS = 0  # Windows Jupyter fix
LR = 1e-3

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True, parents=True)
RESULTS_BASELINE_DIR = PROJECT_ROOT / "results" / "baseline"
RESULTS_BASELINE_DIR.mkdir(exist_ok=True, parents=True)

# Save experiment config
config = {
    "model_name": "BaselineCNN",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "split": "70/15/15 stratified",
    "augmentation": "RandomResizedCrop + HorizontalFlip + Rotation + ColorJitter",
    "loss": "CrossEntropyLoss with class weights",
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "amp": True
}
with open(MODELS_DIR / "baseline_cnn_config.json", "w") as f:
    json.dump(config, f, indent=4)

# Compute class weights from training set
train_labels = train_df["encoded_label"].values
classes_arr = np.unique(train_labels)
class_weights_arr = class_weight.compute_class_weight(
    class_weight="balanced", classes=classes_arr, y=train_labels
)
class_weights_tensor = torch.tensor(class_weights_arr, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# DataLoaders
train_dataset = PlantDiseaseDataset(
    train_df["image_path"].values, train_df["encoded_label"].values, transform=train_tfms
)
val_dataset = PlantDiseaseDataset(
    val_df["image_path"].values, val_df["encoded_label"].values, transform=val_tfms
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# Model, optimizer, scheduler, scaler
model = BaselineCNN(num_classes=num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler = GradScaler()

best_val_f1 = 0.0
best_preds, best_labels_arr = None, None
best_model_path = MODELS_DIR / "baseline_cnn_best.pth"

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "train_f1": [], "val_f1": []
}

print(f"Training BaselineCNN | {EPOCHS} epochs | batch_size={BATCH_SIZE} | AMP=True")
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}\n")

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc, t_f1 = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    v_loss, v_acc, v_f1, preds, labels_out = validate(model, val_loader, criterion)
    
    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_acc"].append(t_acc)
    history["val_acc"].append(v_acc)
    history["train_f1"].append(t_f1)
    history["val_f1"].append(v_f1)
    
    scheduler.step(v_loss)
    
    print(f"Ep {epoch:2d}/{EPOCHS} | TR Loss: {t_loss:.4f} Acc: {t_acc:.4f} F1: {t_f1:.4f} || VA Loss: {v_loss:.4f} Acc: {v_acc:.4f} F1: {v_f1:.4f}", end="")
    
    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        best_preds = preds
        best_labels_arr = labels_out
        torch.save(model.state_dict(), best_model_path)
        print(f"  ★ saved", end="")
    print()

print(f"\nBest Val Macro-F1: {best_val_f1:.4f}")
print(f"Model saved to: {best_model_path}")

# Save training history
history_df = pd.DataFrame(history)
history_df.index.name = "epoch"
history_df.index = history_df.index + 1
history_df.to_csv(RESULTS_BASELINE_DIR / "metrics.csv")
print(f"Metrics saved to: {RESULTS_BASELINE_DIR / 'metrics.csv'}")

In [ ]:
# Training curves: loss, accuracy, macro-F1
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train Accuracy")
axes[1].plot(epochs_range, history["val_acc"], label="Val Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(epochs_range, history["train_f1"], label="Train Macro-F1")
axes[2].plot(epochs_range, history["val_f1"], label="Val Macro-F1")
axes[2].set_title("Macro-F1 Curves")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Macro-F1")
axes[2].legend()

plt.tight_layout()
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES_DIR / "baseline_training_curves.png", dpi=200)
plt.show()

In [ ]:
# Validation set — Classification Report + Confusion Matrix
print("=== Validation Set Classification Report ===\n")
print(classification_report(best_labels_arr, best_preds, target_names=class_names, zero_division=0))

cm = confusion_matrix(best_labels_arr, best_preds)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — Baseline CNN (Validation Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_confusion_matrix_val.png", dpi=200)
plt.show()

## SECTION 11 — Test Set Evaluation

Load the held-out test set and evaluate the best model saved during training.

In [ ]:
# VRAM cleanup from training
del model, optimizer, scaler, criterion
torch.cuda.empty_cache()
gc.collect()

# Load test set
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
test_dataset = PlantDiseaseDataset(
    test_df["image_path"].values, test_df["encoded_label"].values, transform=val_tfms
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# Load best model
model_eval = BaselineCNN(num_classes=num_classes).to(device)
model_eval.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
model_eval.eval()

# Evaluate on test set
test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test Evaluation"):
        images = images.to(device)
        with autocast(device_type='cuda'):
            outputs = model_eval(images)
        preds = outputs.argmax(dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f"=== Test Set Results ===")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro-F1:  {test_f1:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=class_names, zero_division=0))

# Confusion matrix
cm_test = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(20, 16))
sns.heatmap(cm_test, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — Baseline CNN (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_confusion_matrix_test.png", dpi=200)
plt.show()

# Save test results for ablation table
test_results = {
    "model": "baseline_cnn",
    "val_accuracy": max(history["val_acc"]),
    "val_macro_f1": best_val_f1,
    "test_accuracy": test_acc,
    "test_macro_f1": test_f1
}
pd.DataFrame([test_results]).to_csv(RESULTS_BASELINE_DIR / "test_results.csv", index=False)
print(f"\nTest results saved to: {RESULTS_BASELINE_DIR / 'test_results.csv'}")

# Final cleanup
del model_eval
torch.cuda.empty_cache()
gc.collect()